[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/43_flow_matching.ipynb)

# 🔴 Hard: Conditional Flow Matching

Build a small end-to-end Flow Matching generator. A vector field $v_\theta(x,t)$ transports samples from Gaussian noise $p_0$ to a two-dimensional target distribution $p_1$. Unlike diffusion, sampling solves a deterministic ODE.

For independently paired $x_0\sim p_0$ and $x_1\sim p_1$, sample $t\sim U[0,1]$ and use the linear conditional path

$$x_t=(1-t)x_0+t x_1,\qquad u_t=x_1-x_0.$$

Train the field with $\mathbb E\lVert v_\theta(x_t,t)-u_t\rVert^2$, then integrate $dx/dt=v_\theta(x,t)$ from $t=0$ to $1$. This is **Conditional Flow Matching**; independent pairing is not optimal-transport coupling.

### Implement

```python
def compute_fm_loss(model, x0, x1): ...
def sample_ode(model, x0, steps=50): ...
```

`compute_fm_loss` must sample one time per batch item. `sample_ode` must not mutate `x0` and must use Euler steps of size `1 / steps`.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


def sample_p0(batch_size: int, device=None):
    return torch.randn(batch_size, 2, device=device)


def sample_p1(batch_size: int, device=None):
    t = torch.rand(batch_size, device=device) * (2 * torch.pi)
    r = 2.0 + 0.1 * torch.randn(batch_size, device=device)
    return torch.stack((r * torch.cos(t), r * torch.sin(t) * torch.cos(t)), dim=1)


class VectorFieldNet(nn.Module):
    def __init__(self, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        if t.ndim == 1:
            t = t.unsqueeze(1)
        return self.net(torch.cat((x, t), dim=1))


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE
def compute_fm_loss(model: nn.Module, x0: torch.Tensor, x1: torch.Tensor) -> torch.Tensor:
    pass  # sample t: (B, 1); interpolate x_t; return mean squared velocity error


@torch.no_grad()
def sample_ode(model: nn.Module, x0: torch.Tensor, steps: int = 50) -> torch.Tensor:
    pass  # clone x0, then Euler-integrate model(x, t) from 0 to 1


In [ ]:
# 🧪 Train the complete 2D example
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VectorFieldNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for step in range(1, 2001):
    x0, x1 = sample_p0(512, device), sample_p1(512, device)
    loss = compute_fm_loss(model, x0, x1)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 400 == 0:
        print(f'Step {step}/2000 | loss: {loss.item():.4f}')

generated = sample_ode(model, sample_p0(1_000, device), steps=100).cpu()
target = sample_p1(1_000).cpu()
plt.scatter(target[:, 0], target[:, 1], s=4, alpha=0.5, label='target')
plt.scatter(generated[:, 0], generated[:, 1], s=4, alpha=0.5, label='generated')
plt.axis('equal'); plt.legend(); plt.show()


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('flow_matching')
